hanno.rein@utoronto.ca

Ich benutze Rebound falsch

Lieber Hanno,

Trifon Trifonov hat mich zu der Tagung in Sofia Anfang September eingeladen, zu der du auch kommst.

Ich traue mich aber nur dahin, wenn ich rechtzeitig einen schwerwiegenden Bug aus meiner Software entfernen konnte. Dabei hoffe ich auf deine Hilfe.

Ich nutze in meiner Software Rebound, aber ich scheine es falsch zu benutzen.

Vermutlich mache ich einen fuer dich sehr offensichtlichen Fehler, aber ich konnte den Fehler in wochenlanger Suche nicht finden.

Ich habe dieser Mail ein extrem einfaches Pythonskript beigefuegt, dass den Fehler reproduziert.

Ich bin kein Astrophysiker. Nur ein Informatiker mit zu viel Freizeit. Simon Albrecht von der Uni Aarhus beraet mich bei der Astrophysik.

Ich entwickle eine schnelle, bedienerfreundliche und gut dokumentierte n-body Software, die Videos der Sternsysteme und diverse Auswertungen erzeugen und Orbitparameter mit MCMC und anderen Methoden aus Transitzeitpunkten, Fluxdaten und RV-Daten ermitteln kann.

Das waere ganz fantastisch, wenn du mir helfen wuerdest!

Wir koennen uns auch sehr gerne zu einem Videocall verabreden.

Viele Gruesse aus Kopenhagen
Uli Scheuss


In [1]:
import rebound
import math

# defining some constants
g     = 6.67430e-11          # [m**3/kg/s**2] gravitational constant
au    = 1.495978707e11       # [m]   astronomical unit
m_sun = 1.98847e30           # [kg]  solar mass
m_jup = 1.8981246e27         # [kg]  Jupiter mass
deg   = 1 / 180 * math.pi
day   = 24 * 60 * 60         # [s]
T0    = 2458400.0            # Epoch
TTc00 = 2458401.4086         # observed TT Number  0 of TOI-4504c
TTc31 = 2460970.0865         # observed TT Number 31 of TOI-4504c

In [2]:
# setting up the simulation
sim1 = rebound.Simulation()
sim1.G = g

name = "TOI-4504"   # star
mass = 0.885 * m_sun
sim1.add(hash=name, m=mass)

name = "TOI-4504d"  # inner warm Jupiter
mass =      2.1278  * m_jup
e =         0.2498  * 1
i =         91.10   * deg
P =         41.7878 * day
Omega =     0.24    * deg
omega =     275.80  * deg
ma =        339.54  * deg
sim1.add(hash=name, m=mass, P=P, inc=i, e=e, Omega=Omega, omega=omega, M=ma)

name = "TOI-4504c"  # outer warm Jupiter
mass =      2.6315  * m_jup
e =         0.0377  * 1
i =         89.69   * deg
P =         81.4631 * day
Omega =     0.8     * deg
omega =     298.89  * deg
ma =        142.72  * deg
sim1.add(hash=name, m=mass, P=P, inc=i, e=e, Omega=Omega, omega=omega, M=ma)

sim1.move_to_com()

With these orbital parameters the simulation reproduces all observed transits (14 planet c and 6 planet d) of the TOI4504 system well.

In the next cell, the simulation runs and confirms the first and the last transit of the outer planet c by checking that dx changes sign.

In [3]:
# running the simulation
delta = 0.01
TOI4504  = sim1.particles[0]
TOI4504d = sim1.particles[1]
TOI4504c = sim1.particles[2]

sim1.integrate((TTc00 - T0 - delta) * day)
print(f"dx {delta} days before first transit of TOI4504c at {TTc00}: {TOI4504c.x - TOI4504.x:13_.0f} m")
sim1.integrate((TTc00 - T0 + delta) * day)
print(f"dx {delta} days after  first transit of TOI4504c at {TTc00}: {TOI4504c.x - TOI4504.x:13_.0f} m")
sim1.integrate((TTc31 - T0 - delta) * day)
# print(f"dx before TTc31: {TOI4504c.x - TOI4504.x:13_.0f} m")
print(f"dx {delta} days before last  transit of TOI4504c at {TTc31}: {TOI4504c.x - TOI4504.x:13_.0f} m")
sim1.integrate((TTc31 - T0 + delta) * day)
# print(f"dx after TTc31:  {TOI4504c.x - TOI4504.x:13_.0f} m")
print(f"dx {delta} days after  last  transit of TOI4504c at {TTc31}: {TOI4504c.x - TOI4504.x:13_.0f} m")

dx 0.01 days before first transit of TOI4504c at 2458401.4086:    49_205_446 m
dx 0.01 days after  first transit of TOI4504c at 2458401.4086:   -29_471_875 m
dx 0.01 days before last  transit of TOI4504c at 2460970.0865:    60_898_891 m
dx 0.01 days after  last  transit of TOI4504c at 2460970.0865:   -20_201_470 m


Even so, the orbital parameters of sim1 are incorrect!

Two papers on this system and my own experiments with the ttvfast library all agree that instead, the orbital parameters in the following simulation sim2 are correct:

In [4]:
sim2 = rebound.Simulation()
sim2.G = g

name = "TOI-4504"   # star
mass = 0.885 * m_sun
sim2.add(hash=name, m=mass)

name = "TOI-4504d"  # inner warm Jupiter
mass =      2.1512    * m_jup
e =         0.2502    * 1
i =         88.76     * deg
P =         41.7819   * day
Omega =     0.0       * deg
omega =     276.77    * deg
ma =        337.16    * deg
sim2.add(hash=name, m=mass, P=P, inc=i, e=e, Omega=Omega, omega=omega, M=ma)

name = "TOI-4504c"  # outer warm Jupiter
mass =      2.6385    * m_jup
e =         0.0350    * 1
i =         89.71     * deg
P =         81.820097 * day
Omega =    -2.016956  * deg
omega =     303.62    * deg
ma =        137.89    * deg
sim2.add(hash=name, m=mass, P=P, inc=i, e=e, Omega=Omega, omega=omega, M=ma)

sim2.move_to_com()

In [5]:
delta = 0.01
TOI4504  = sim2.particles[0]
TOI4504d = sim2.particles[1]
TOI4504c = sim2.particles[2]

sim2.integrate((TTc00 - T0 - delta) * day)
print(f"dx {delta} days before first transit of TOI4504c at {TTc00}: {TOI4504c.x - TOI4504.x:13_.0f} m")
sim2.integrate((TTc00 - T0 + delta) * day)
print(f"dx {delta} days after  first transit of TOI4504c at {TTc00}: {TOI4504c.x - TOI4504.x:13_.0f} m")
sim2.integrate((TTc31 - T0 - delta) * day)
# print(f"dx before TTc31: {TOI4504c.x - TOI4504.x:13_.0f} m")
print(f"dx {delta} days before last  transit of TOI4504c at {TTc31}: {TOI4504c.x - TOI4504.x:13_.0f} m")
sim2.integrate((TTc31 - T0 + delta) * day)
# print(f"dx after TTc31:  {TOI4504c.x - TOI4504.x:13_.0f} m")
print(f"dx {delta} days after  last  transit of TOI4504c at {TTc31}: {TOI4504c.x - TOI4504.x:13_.0f} m")

dx 0.01 days before first transit of TOI4504c at 2458401.4086:    57_359_304 m
dx 0.01 days after  first transit of TOI4504c at 2458401.4086:   -21_467_970 m
dx 0.01 days before last  transit of TOI4504c at 2460970.0865: 19_594_644_256 m
dx 0.01 days after  last  transit of TOI4504c at 2460970.0865: 19_517_906_654 m


Aber mit den korrekten Parametern erhalte ich eine falsche Simulation.

Der erste Transit passt noch einigermassen, aber vor allem aufgrund der zu kurzen Periode des aeusseren Planeten kommt der letzte Transit 5 Tage zu frueh.

Abgesehen von der Periode des aeusseren Planeten weichen die anderen Parameter von sim1 und sim2 eher geringfuegig voneinander ab.

Das legt nahe, dass es sich hier um ein Problem zwischen Jacobi Koordinaten und astrozentrischen Koordinaten handeln koennte, aber soweit ich erkennen kann, liegt es daran nicht. Ich glaube, haette ich in sim1 astrozentrische Koordinaten benutzt, waere die Periode fuer TOI4504c sogar noch kuerzer, etwa 80.76 Tage.

Ich kann mit sim2 aber die observed TT auch nicht dadurch reproduzieren, dass ich lediglich P(c) anpasse.

Eines der Paper benutzt Rebound.

Alle Simulationen von mir und in den verglichenen Papern nutzen Jacobi Koordinaten.

Aber irgendwas mache ich falsch. Kannst du sehen, was?

Der Uebersichtlichkeit halber habe ich in der folgenden Tabelle noch einmal die beiden Parametersaetze gegenuebergestellt.

sim1 erzeugt mit falschen Parametern die richtigen Transit Times.

sim2 erzeugt mit richtigen Parametern die falschen Transit Times.

| Body     | Param | Unit   | sim1     | sim2      |
|----------|-------|--------|----------|-----------|
| TOI4504  | mass  | m_sun  | 0.885    | 0.885     |
| TOI4504d | mass  | m_jup  | 2.1278   | 2.1512    |
| TOI4504d | e     |        | 0.2498   | 0.2502    |
| TOI4504d | i     | deg    | 91.10    | 88.76     |
| TOI4504d | P     | d      | 41.7878  | 41.7819   |
| TOI4504d | Omega | deg    | 0.24     | 0         |
| TOI4504d | omega | deg    | 275.80   | 276.77    |
| TOI4504d | ma    | deg    | 339.54   | 337.16    |
| TOI4504c | mass  | m_jup  | 2.6315   | 2.6385    |
| TOI4504c | e     |        | 0.0377   | 0.0350    |
| TOI4504c | i     | deg    | 89.69    | 89.71     |
| TOI4504c | P     | d      | 81.4631  | 81.820097 |
| TOI4504c | Omega | deg    | 0.8      | -2.016956 |
| TOI4504c | omega | deg    | 298.9    | 303.62    |
| TOI4504c | ma    | deg    | 142.7    | 137.89    |